# COS711 Assignment 2: Segmentation Enhancement

This notebook starts from notebook 02's final model (ResNet-34 U-Net with five outputs, `C_final`, three seeds) and asks one question per section: **what can be done beyond the standard pipeline, and how much does each idea help?**

| Section | Idea | Error it targets |
|---|---|---|
| 3 | What goes wrong with the baseline | (analysis) |
| 4 | Instance extraction that uses all five output maps: hysteresis, fragment joining, contact/distance splitting, rule-based and learned instance filtering | false positives, fragmented filaments, merged filaments |
| 5 | Better probabilities: test-time augmentation (8 rotations/reflections) and the 3-seed ensemble | everything, at a compute cost |
| 6 | Compression: fp16 vs fp32, half-resolution inference, unstructured and structured pruning (with fine-tuning), knowledge distillation into a smaller student | computational cost |
| 7-8 | Choosing the final pipeline (accuracy vs. end-to-end time), test predictions and `submission.csv` | |

**Protocol.** Notebook 02 tuned the threshold and minimum area on the same validation set it reported (0.388 → 0.405), which is optimistic. Here every post-processing choice is **cross-fitted**: the 107 validation observations are split into two folds by whole 7-day blocks, settings are tuned on one fold and scored on the other, and only those out-of-fold scores are reported. Every setting is then also scored on the outputs of all three seeds (mean ± std), and gains get a 95% bootstrap interval over observations.

**How to run:** see `docs/kaggle_guide.md`. In short: Kaggle notebook, GPU T4, Internet on, add the competition data and the notebook-02 output that holds `runs/C_final_s*/best.pt`, then *Save Version → Save & Run All*. Everything is written to `/kaggle/working/enhancement`, and a rerun skips work that is already there.

## 1. Setup

In [ ]:
# Clone the team repo (notebook 01, src/) exactly like notebook 02 does.
import os, subprocess, sys, time, shutil, json, copy
from pathlib import Path
T_START = time.time()
ON_KAGGLE = os.path.exists("/kaggle/input")
REPO_URL, BRANCH = "https://github.com/Sean-Mp/COS711-Solar-Filament.git", "chisom/experimental-setup"
REPO = "/kaggle/temp/repo" if ON_KAGGLE else "/content/repo"
if not os.path.exists(REPO):
    assert subprocess.run(["git", "clone", "-q", REPO_URL, REPO]).returncode == 0, (
        "git clone failed. On Kaggle, turn Internet on (right-hand panel → Session options → Internet).")
subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
sys.path.insert(0, REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")   # for worker processes

!pip -q install segmentation-models-pytorch torch-pruning

### Enhancement code

The next cells write this section's modules into `src/` (they are the same files as `src/*.py` in the repo, so the notebook runs even before they are pushed):

* `metrics.py`: notebook 01's PQ and split/merge counts, importable by worker processes
* `postprocess.py`: the four-stage instance extraction (hysteresis → join → split → filter)
* `inference.py`: TTA, ensembles, reduced resolution, probability caching
* `enh_eval.py`: cross-fitted tuning chain, multi-seed scoring, bootstrap, timing, figures
* `compress.py`: unstructured/structured pruning, size measurements, distillation loss
* `submission.py`: run-length encoding and the submission writer

In [ ]:
%%writefile src/metrics.py
"""Instance metrics, identical to the ones defined in notebook 01 (panoptic_quality, split_merge).

They are copied here so worker processes and the enhancement modules can import them without
running notebook 01. Notebook 03 checks that both versions give the same scores.
"""
import numpy as np


def panoptic_quality(pred_inst, gt_inst, iou_thr=0.5):
    """PQ = SQ * RQ. IoU>0.5 guarantees unique matches"""
    P, G = pred_inst.max() + 1, gt_inst.max() + 1
    cont = np.bincount(gt_inst.ravel().astype(np.int64) * P + pred_inst.ravel(),
                       minlength=G * P).reshape(G, P)
    ga, pa = cont.sum(1), cont.sum(0)
    iou = cont / (ga[:, None] + pa[None, :] - cont + 1e-9)
    iou[0, :] = 0; iou[:, 0] = 0                   # ignore background
    gi, pi = np.nonzero(iou > iou_thr)
    tp = len(gi); ious = iou[gi, pi]
    n_gt, n_pr = len(np.unique(gt_inst)) - (0 in gt_inst), len(np.unique(pred_inst)) - (0 in pred_inst)
    fp, fn = n_pr - tp, n_gt - tp
    sq = ious.mean() if tp else 0.0
    rq = tp / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) else 1.0
    return dict(PQ=sq * rq, SQ=sq, RQ=rq, TP=tp, FP=fp, FN=fn)


def split_merge(pred_inst, gt_inst, min_frac=0.1):
    """one-to-many: GT covered by >=2 preds; many-to-one: pred covering >=2 GTs.
    A link counts if the intersection is >= min_frac of the *smaller* object's area."""
    P, G = pred_inst.max() + 1, gt_inst.max() + 1
    cont = np.bincount(gt_inst.ravel().astype(np.int64) * P + pred_inst.ravel(),
                       minlength=G * P).reshape(G, P)[1:, 1:]
    if cont.size == 0: return dict(one_to_many=0, many_to_one=0, n_gt=G-1, n_pred=P-1)
    ga, pa = cont.sum(1), cont.sum(0)
    link = cont >= min_frac * np.minimum(ga[:, None], pa[None, :]).clip(1)
    link &= cont > 0
    return dict(one_to_many=int((link.sum(1) >= 2).sum()),
                many_to_one=int((link.sum(0) >= 2).sum()),
                n_gt=int((ga > 0).sum()), n_pred=int((pa > 0).sum()))


def best_iou_per_pred(pred_inst, gt_inst):
    "For each predicted instance 1..P-1, its highest IoU with any ground-truth instance."
    P, G = int(pred_inst.max()) + 1, int(gt_inst.max()) + 1
    cont = np.bincount(gt_inst.ravel().astype(np.int64) * P + pred_inst.ravel(),
                       minlength=G * P).reshape(G, P)
    ga, pa = cont.sum(1), cont.sum(0)
    iou = cont / (ga[:, None] + pa[None, :] - cont + 1e-9)
    iou[0, :] = 0
    return iou[:, 1:].max(0) if G > 1 else np.zeros(P - 1)

In [ ]:
%%writefile src/postprocess.py
"""Segmentation Enhancement: turning the U-Net's five output maps into filament instances.

Every public pipeline has the same signature as the baseline `evaluate.instances_cc`:

    fn(probs, disk, **kwargs) -> int32 instance map (0 = background, 1..n = filaments)

so it plugs into `evaluate.score_collected`, `evaluate.grid_search` and `evaluate.benchmark`.
`probs` is (5, H, W) float32 in [0, 1], in the channel order of `unet.TARGET_CHANNELS`:
fg, boundary, contact, dist, spine.

`enhanced` chains four optional stages, each aimed at one error type the baseline makes:

1. foreground   plain threshold, or hysteresis: keep low-confidence pixels only when they are
                connected to high-confidence ones (fewer faint false positives, fewer gaps)
2. join         group fragments of one filament: pieces bridged by low-confidence fg, by the
                predicted spine, or by a small morphological closing get the same label
                (targets fragmentation, the "one-to-many" error)
3. split        cut instances along predicted contact / boundary lines, or between distance-map
                peaks, with a marker-controlled watershed (targets merges, "many-to-one")
4. filter       drop instances by size, mean confidence, position on the disk, or by a small
                classifier trained on instance features (targets false positives)

With every option at its default, `enhanced(probs, disk, thr, min_area)` reproduces the baseline
connected-components extraction exactly.
"""
import cv2
import numpy as np
from skimage.segmentation import watershed

FG, BOUNDARY, CONTACT, DIST, SPINE = range(5)

FEATURES = ["log_area", "mean_fg", "max_fg", "frac_hi", "mean_spine", "max_spine", "mean_boundary",
            "mean_contact", "mean_dist", "max_dist", "elongation", "log_length", "mean_width", "rdist"]


# ----------------------------------------------------------------------------- helpers

def _kernel(r):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))


def label_cc(mask):
    "8-connected components of a boolean mask, as int32 labels."
    _, lab = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8, ltype=cv2.CV_32S)
    return lab


def relabel(lab, keep=None):
    """Consecutive labels 1..n. `keep` (bool per old label, index 0 ignored) drops instances."""
    m = int(lab.max())
    if m == 0:
        return lab.astype(np.int32)
    present = np.bincount(lab.ravel(), minlength=m + 1) > 0
    present[0] = False
    if keep is not None:
        present &= np.asarray(keep, bool)
    lut = np.zeros(m + 1, np.int32)
    lut[present] = np.arange(1, present.sum() + 1, dtype=np.int32)
    return lut[lab]


def disk_geometry(disk):
    "Centre (cx, cy) and radius of the solar disk mask, from its image moments."
    mo = cv2.moments(disk.astype(np.uint8), binaryImage=True)
    if mo["m00"] == 0:
        h, w = disk.shape
        return w / 2, h / 2, min(h, w) / 2
    return mo["m10"] / mo["m00"], mo["m01"] / mo["m00"], float(np.sqrt(mo["m00"] / np.pi))


# ----------------------------------------------------------------------------- stage 1: foreground

def foreground(probs, disk, thr=0.5, lo=None, min_seed=0):
    """Pixels that belong to some filament.

    lo=None: fg > thr (the baseline). lo < thr: hysteresis, i.e. connected regions of fg > lo
    that contain at least `min_seed` pixels (and at least one) with fg > thr."""
    fg = probs[FG]
    if lo is None or lo >= thr:
        return (fg > thr) & disk, None
    lab = label_cc((fg > lo) & disk)
    n = int(lab.max())
    strong = np.bincount(lab[(fg > thr) & disk], minlength=n + 1)
    keep = strong >= max(int(min_seed), 1)
    keep[0] = False
    return keep[lab], None


# ----------------------------------------------------------------------------- stage 2: join

def join_fragments(lab, probs, disk, join_r=0, join_thr=None, spine_thr=None):
    """Give fragments of one filament the same label, without adding pixels to the mask.

    Fragments are grouped if they are connected through (a) fg > join_thr, (b) spine > spine_thr,
    or (c) a morphological closing of radius join_r (gaps up to about 2 * join_r pixels)."""
    if not join_r and join_thr is None and spine_thr is None:
        return lab
    bridge = lab > 0
    if join_thr is not None:
        bridge |= (probs[FG] > join_thr) & disk
    if spine_thr is not None:
        bridge |= (probs[SPINE] > spine_thr) & disk
    if join_r:
        bridge = cv2.morphologyEx(bridge.astype(np.uint8), cv2.MORPH_CLOSE, _kernel(int(join_r))) > 0
    groups = label_cc(bridge)
    return relabel(np.where(lab > 0, groups, 0))


# ----------------------------------------------------------------------------- stage 3: split

def split_instances(lab, probs, split=None, cut_thr=0.5, d_thr=0.5, min_part=50):
    """Split instances that the network marks as two touching filaments.

    split="contact" / "boundary": remove pixels with contact (boundary) > cut_thr, split="dist":
    keep only pixels with dist > d_thr. The remaining connected parts of at least `min_part`
    pixels become watershed markers; an instance is split only if it holds two or more markers,
    and its pixels are then assigned to the markers by a watershed on the boundary map."""
    if split is None:
        return lab
    mask = lab > 0
    if split == "contact":
        core = mask & (probs[CONTACT] <= cut_thr)
    elif split == "boundary":
        core = mask & (probs[BOUNDARY] <= cut_thr)
    elif split == "dist":
        core = mask & (probs[DIST] > d_thr)
    else:
        raise ValueError(f"unknown split mode {split!r}")
    markers = label_cc(core)
    nm = int(markers.max())
    if nm < 2:
        return lab
    size = np.bincount(markers.ravel(), minlength=nm + 1)
    ok = size >= min_part
    ok[0] = False
    markers = relabel(markers, ok)
    nm = int(markers.max())
    if nm < 2:
        return lab
    # markers per instance (each marker lies inside exactly one instance)
    sel = markers > 0
    pairs = np.unique(np.stack([lab[sel], markers[sel]]), axis=1)
    per_inst = np.bincount(pairs[0], minlength=int(lab.max()) + 1)
    to_split = per_inst >= 2
    if not to_split.any():
        return lab
    region = to_split[lab]
    elev = (probs[BOUNDARY] * 255).astype(np.uint8)
    ws = watershed(elev, np.where(region, markers, 0), mask=region)
    out = np.where(region & (ws > 0), ws + int(lab.max()), lab)
    return relabel(out)


# ----------------------------------------------------------------------------- stage 4: filter

def instance_features(lab, probs, disk):
    "One row of FEATURES per instance 1..n (row i-1 is instance i), plus the raw area."
    n = int(lab.max())
    if n == 0:
        return np.zeros((0, len(FEATURES)), np.float32), np.zeros(0)
    yy, xx = np.nonzero(lab)
    ll = lab[yy, xx]
    area = np.bincount(ll, minlength=n + 1)[1:].astype(np.float64)
    a = np.maximum(area, 1)

    def mean_of(ch):
        return np.bincount(ll, probs[ch][yy, xx], minlength=n + 1)[1:] / a

    def max_of(ch):
        out = np.zeros(n + 1)
        np.maximum.at(out, ll, probs[ch][yy, xx])
        return out[1:]

    fgv = probs[FG][yy, xx]
    frac_hi = np.bincount(ll, (fgv > 0.7).astype(np.float64), minlength=n + 1)[1:] / a
    # second moments -> elongation and length of the principal axis
    sx = np.bincount(ll, xx, minlength=n + 1)[1:] / a
    sy = np.bincount(ll, yy, minlength=n + 1)[1:] / a
    sxx = np.bincount(ll, xx.astype(np.float64) ** 2, minlength=n + 1)[1:] / a - sx ** 2
    syy = np.bincount(ll, yy.astype(np.float64) ** 2, minlength=n + 1)[1:] / a - sy ** 2
    sxy = np.bincount(ll, xx.astype(np.float64) * yy, minlength=n + 1)[1:] / a - sx * sy
    tr, det = sxx + syy, sxx * syy - sxy ** 2
    disc = np.sqrt(np.maximum(tr ** 2 / 4 - det, 0))
    l1, l2 = tr / 2 + disc, np.maximum(tr / 2 - disc, 0)
    length = 4 * np.sqrt(np.maximum(l1, 0)) + 1
    elong = np.sqrt((l1 + 1) / (l2 + 1))
    cx, cy, r = disk_geometry(disk)
    rdist = np.hypot(sx - cx, sy - cy) / max(r, 1)
    X = np.stack([np.log(a), mean_of(FG), max_of(FG), frac_hi, mean_of(SPINE), max_of(SPINE),
                  mean_of(BOUNDARY), mean_of(CONTACT), mean_of(DIST), max_of(DIST), elong,
                  np.log(length), area / length, rdist], 1).astype(np.float32)
    return X, area


def filter_instances(lab, probs, disk, min_area=0, min_mean=0.0, max_rdist=None, clf=None, clf_thr=0.5):
    """Drop instances smaller than min_area, with mean fg probability below min_mean, whose
    centroid lies beyond max_rdist disk radii, or that the classifier `clf` (anything with
    predict_proba on FEATURES) scores below clf_thr."""
    n = int(lab.max())
    if n == 0 or (not min_area and not min_mean and max_rdist is None and (clf is None or not clf_thr)):
        return relabel(lab)
    if not min_mean and max_rdist is None and (clf is None or not clf_thr):
        area = np.bincount(lab.ravel(), minlength=n + 1)[1:]     # size only: no features needed
        return relabel(lab, np.r_[False, area >= min_area])
    X, area = instance_features(lab, probs, disk)
    keep = np.ones(n, bool)
    if min_area:
        keep &= area >= min_area
    if min_mean:
        keep &= X[:, FEATURES.index("mean_fg")] >= min_mean
    if max_rdist is not None:
        keep &= X[:, FEATURES.index("rdist")] <= max_rdist
    if clf is not None and clf_thr:
        keep &= clf.predict_proba(X)[:, 1] >= clf_thr
    return relabel(lab, np.r_[False, keep])


# ----------------------------------------------------------------------------- full pipeline

FILTER_KEYS = ("min_area", "min_mean", "max_rdist", "clf", "clf_thr")


def upstream(probs, disk, thr=0.5, lo=None, min_seed=0, join_r=0, join_thr=None, spine_thr=None,
             split=None, cut_thr=0.5, d_thr=0.5, min_part=50):
    "Stages 1-3: foreground -> connected components -> join fragments -> split merges."
    fg, _ = foreground(probs, disk, thr, lo, min_seed)
    lab = label_cc(fg)
    lab = join_fragments(lab, probs, disk, join_r, join_thr, spine_thr)
    return split_instances(lab, probs, split, cut_thr, d_thr, min_part)


def enhanced(probs, disk, min_area=50, min_mean=0.0, max_rdist=None, clf=None, clf_thr=0.5, **kw):
    "The full pipeline: upstream stages, then the instance filter (stage 4)."
    return filter_instances(upstream(probs, disk, **kw), probs, disk, min_area, min_mean, max_rdist, clf, clf_thr)

In [ ]:
%%writefile src/inference.py
"""Prediction variants for the enhancement study, and on-disk caching of their probability maps.

* test-time augmentation over the 8 rotations/reflections of the square (tta=8), or the 4 flips
  already used by evaluate.predict_probs (tta=4)
* ensembles: the probabilities of several models (e.g. the three training seeds) are averaged
* reduced-resolution inference (scale < 1): the network runs on a downscaled disk and the
  probabilities are upsampled again, trading accuracy for about 1/scale^2 less network compute
* amp=False runs the network in fp32 (the default is fp16 autocast on GPU, as in notebook 02)

Cached maps are stored as uint8 (probability * 255), one compressed .npz per observation, so the
post-processing experiments can run on CPU without the network.
"""
import json
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from . import evaluate as ev

FLIPS4 = [(0, False), (0, True), (2, False), (2, True)]           # = evaluate.predict_probs(tta=True)
DIHEDRAL8 = [(k, f) for f in (False, True) for k in range(4)]


def _tfm(x, k, f):
    x = torch.rot90(x, k, (2, 3)) if k else x
    return x.flip(3) if f else x


def _inv(x, k, f):
    x = x.flip(3) if f else x
    return torch.rot90(x, -k, (2, 3)) if k else x


@torch.no_grad()
def predict_probs(models, img, tta=1, amp=True, scale=1.0, multiple=32):
    """img (H, W) float32 in [0, 1] -> (C, H, W) float32 mean sigmoid probability over the models
    and the test-time transforms. With one model, tta=1 and scale=1 this equals
    evaluate.predict_probs(model, img)."""
    models = models if isinstance(models, (list, tuple)) else [models]
    dev = next(models[0].parameters()).device
    H, W = img.shape
    src = img
    if scale != 1.0:
        src = cv2.resize(img, (round(W * scale), round(H * scale)), interpolation=cv2.INTER_AREA)
    h, w = src.shape
    x = torch.from_numpy(np.ascontiguousarray(src, np.float32))[None, None].to(dev)
    x = F.pad(x, (0, -w % multiple, 0, -h % multiple))
    tfms = {1: [(0, False)], 4: FLIPS4, 8: DIHEDRAL8}[tta]
    p, n = 0, 0
    with torch.autocast(dev.type, dtype=torch.float16, enabled=amp and dev.type == "cuda"):
        for m in models:
            m.eval()
            for k, f in tfms:
                out = torch.sigmoid(m(_tfm(x, k, f)).float())
                p = p + _inv(out, k, f)
                n += 1
    p = (p / n)[0, :, :h, :w].cpu().numpy()
    if scale != 1.0:
        p = cv2.resize(p.transpose(1, 2, 0), (W, H), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        p = np.ascontiguousarray(p)
    return p.astype(np.float32)


def prob_path(prob_dir, obs):
    return Path(prob_dir) / f"{obs}.npz"


def load_probs(prob_dir, obs):
    return np.load(prob_path(prob_dir, obs))["q"].astype(np.float32) / 255


def cache_probs(predict_fn, items, cache, out_dir, progress=None):
    """Run predict_fn(img) -> (C, H, W) probabilities on every item's full disk and save uint8
    maps to out_dir. Observations already cached are skipped. Writes out_dir/timing.json with
    the mean network seconds per observation (GPU synchronised by the .cpu() inside)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    times = []
    todo = [it for it in items if not prob_path(out_dir, it["obs"]).exists()]
    for it in (progress(todo) if progress else todo):
        img, _ = ev.load_obs(cache, it["obs"])
        t0 = time.perf_counter()
        p = predict_fn(img)
        times.append(time.perf_counter() - t0)
        np.savez_compressed(prob_path(out_dir, it["obs"]), q=(p * 255 + 0.5).astype(np.uint8))
    tfile = out_dir / "timing.json"
    if times:
        old = json.loads(tfile.read_text()) if tfile.exists() else {}
        n_old = old.get("n", 0)
        mean = (old.get("network_s", 0) * n_old + sum(times)) / (n_old + len(times))
        tfile.write_text(json.dumps(dict(network_s=mean, n=n_old + len(times))))
    return json.loads(tfile.read_text()) if tfile.exists() else {}


def average_cached(src_dirs, out_dir, items):
    "Ensemble on CPU: average the cached uint8 maps of several variants (e.g. seeds)."
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for it in items:
        f = prob_path(out_dir, it["obs"])
        if f.exists():
            continue
        acc = sum(np.load(prob_path(d, it["obs"]))["q"].astype(np.float32) for d in src_dirs)
        np.savez_compressed(f, q=(acc / len(src_dirs) + 0.5).astype(np.uint8))
    t = [json.loads((Path(d) / "timing.json").read_text()).get("network_s", np.nan)
         for d in src_dirs if (Path(d) / "timing.json").exists()]
    if t:
        (out_dir / "timing.json").write_text(json.dumps(dict(network_s=float(np.sum(t)), n=len(items),
                                                             note="sum of the members' times")))
    return out_dir

In [ ]:
%%writefile src/enh_eval.py
"""Evaluation protocol for the Segmentation Enhancement section.

Problem with tuning post-processing on the validation set and reporting the same set (as in
notebook 02, 0.388 -> 0.405): the gain is optimistic. Here every post-processing setting is
**cross-fitted**: the 107 validation observations are split into two folds by whole 7-day
blocks (the same blocking as the train/validation split, so neighbouring days never straddle
folds). Settings are tuned on one fold and scored on the other, and the two held-out halves are
pooled. That out-of-fold (OOF) score is the number to report.

The ablation is a greedy chain of steps (threshold -> hysteresis -> join -> split -> filter ...).
Each step tunes only its own parameters, with the earlier steps fixed to what *that fold's*
chain chose, and each grid contains the "do nothing" setting, so a step that does not help can
be switched off. A third context, "all", tunes the same chain on every validation observation;
its parameters are the ones used on the test set.

Scores use notebook 01's protocol (evaluate.score / evaluate.summarise): each observation is
scored against every annotation set, averaged over sets, then over observations.
"""
import itertools
import time
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from . import evaluate as ev
from . import postprocess as pp
from .inference import load_probs
from .metrics import best_iou_per_pred, panoptic_quality, split_merge


# ----------------------------------------------------------------------------- folds

def make_folds(items, obs_meta, k=2, block_days=7, seed=0):
    "obs -> fold (0..k-1). Whole 7-day blocks go to one fold; folds get similar numbers of obs."
    obs = [it["obs"] for it in items]
    if "date" in obs_meta:
        dates = pd.to_datetime(obs_meta.set_index("obs").loc[obs, "date"]).reset_index(drop=True)
    else:
        dates = pd.to_datetime(pd.Series([o[:14] for o in obs]), format="%Y%m%d%H%M%S")
    block = ((dates - dates.min()).dt.days // block_days).to_numpy()
    size = pd.Series(block).value_counts()
    rng = np.random.default_rng(seed)
    count = np.zeros(k)
    fold_of_block = {}
    for b in rng.permutation(size.index.to_numpy()):
        f = int(np.argmin(count))
        fold_of_block[b] = f
        count[f] += size[b]
    return {o: fold_of_block[b] for o, b in zip(obs, block)}


# ----------------------------------------------------------------------------- scoring

def _eval_obs(prob_dir, cache, it, postproc, configs):
    probs = load_probs(prob_dir, it["obs"])
    _, disk = ev.load_obs(cache, it["obs"])
    gts = ev.load_gts(cache, it["image_ids"])
    any_gt = np.logical_or.reduce([g > 0 for g in gts])
    memo = {}   # pp.enhanced: configs that differ only in the filter reuse stages 1-3
    rows = []
    for cid, kw in configs:
        t0 = time.perf_counter()
        if postproc is pp.enhanced:
            up_kw = {k: v for k, v in kw.items() if k not in pp.FILTER_KEYS}
            key = tuple(sorted((k, repr(v)) for k, v in up_kw.items()))
            if key not in memo:
                t_up = time.perf_counter()
                memo[key] = (pp.upstream(probs, disk, **up_kw), time.perf_counter() - t_up)
            lab, up_s = memo[key]
            t0 = time.perf_counter() - up_s          # charge the shared stages to every config
            pred = pp.filter_instances(lab, probs, disk, **{k: kw[k] for k in pp.FILTER_KEYS if k in kw})
        else:
            pred = postproc(probs, disk, **kw)
        dt = time.perf_counter() - t0
        # Scoring only needs pixels that are filament in the prediction or in some annotation;
        # PQ, split/merge counts and Dice are unchanged, and it is ~50x faster than full images.
        sel = (pred > 0) | any_gt
        for r in ev.score(pred[sel], [g[sel] for g in gts], it["image_ids"], it["obs"],
                          panoptic_quality, split_merge):
            r.update(cid=cid, pp_s=dt)
            rows.append(r)
    return rows


def evaluate_configs(prob_dir, items, cache, configs_for, postproc=pp.enhanced, n_jobs=4):
    """configs_for(obs) -> list of (config id, kwargs). Each observation's probabilities are loaded
    once and every config is run on them. Returns one row per (config, annotation set)."""
    jobs = (delayed(_eval_obs)(str(prob_dir), str(cache), it, postproc, configs_for(it["obs"]))
            for it in items)
    rows = Parallel(n_jobs=n_jobs)(jobs) if n_jobs != 1 else [j[0](*j[1], **j[2]) for j in jobs]
    return pd.DataFrame([r for rs in rows for r in rs])


def score_params(prob_dir, items, cache, params_for, postproc=pp.enhanced, n_jobs=4):
    """Score each observation with params_for(obs) (a kwargs dict, or one dict for all).
    Returns (summary dict, per-set DataFrame) like evaluate.score_collected."""
    f = params_for if callable(params_for) else (lambda o: params_for)
    df = evaluate_configs(prob_dir, items, cache, lambda o: [(0, f(o))], postproc, n_jobs)
    return summary(df), df


def summary(df):
    s = ev.summarise(df.drop(columns=["cid"], errors="ignore"))
    if "pp_s" in df:
        s["pp_s"] = float(df.groupby("obs").pp_s.first().mean())
    return s


def per_obs_pq(df):
    return df.groupby("obs").PQ.mean()


# ----------------------------------------------------------------------------- grids and chain

def _valid(kw):
    lo, thr = kw.get("lo"), kw.get("thr", 0.5)
    if lo is not None and lo >= thr:
        return False
    if kw.get("join_thr") is not None and kw["join_thr"] >= thr:
        return False
    return True


KEEP = "keep"   # grid value meaning "leave this parameter as the earlier steps chose it"


def expand(grid, base):
    """Configs for one step, merged onto base (the earlier steps' choices). grid is either a dict
    of lists (every combination) or a list of dicts (exactly these). Invalid and duplicate
    configs are dropped; order is kept, so list the "do nothing" setting first."""
    combos = grid if isinstance(grid, list) else (
        [dict(zip(grid, v)) for v in itertools.product(*grid.values())] if grid else [{}])
    out, seen = [], set()
    for over in combos:
        kw = {**base, **{k: v for k, v in over.items() if not (isinstance(v, str) and v == KEEP)}}
        key = tuple(sorted((k, repr(v)) for k, v in kw.items() if k != "clf"))
        if _valid(kw) and key not in seen:
            seen.add(key)
            out.append(kw)
    return out


def _instances_obs(prob_dir, cache, it, postproc, kw):
    probs = load_probs(prob_dir, it["obs"])
    _, disk = ev.load_obs(cache, it["obs"])
    pred = postproc(probs, disk, **kw)
    X, area = pp.instance_features(pred, probs, disk)
    if len(X) == 0:
        return pd.DataFrame()
    gts = ev.load_gts(cache, it["image_ids"])
    ious = np.stack([best_iou_per_pred(pred, g) for g in gts])
    df = pd.DataFrame(X, columns=pp.FEATURES)
    df.insert(0, "inst", np.arange(1, len(X) + 1))
    df.insert(0, "obs", it["obs"])
    df["area"] = area
    df["match_frac"] = (ious > 0.5).mean(0)   # share of annotators for whom it is a true positive
    df["best_iou"] = ious.mean(0)
    return df


def instance_table(prob_dir, items, cache, params_for, postproc=pp.enhanced, n_jobs=4):
    "One row per predicted instance: FEATURES, area, and whether annotators count it as a match."
    f = params_for if callable(params_for) else (lambda o: params_for)
    dfs = Parallel(n_jobs=n_jobs)(delayed(_instances_obs)(str(prob_dir), str(cache), it, postproc, f(it["obs"]))
                                  for it in items)
    dfs = [d for d in dfs if len(d)]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=["obs", "inst", *pp.FEATURES, "area", "match_frac", "best_iou"])


def fit_filter(table, seed=0):
    """Small gradient-boosted classifier: is this predicted instance a real filament (a match for
    at least half of the annotators)? Trained on instance FEATURES only."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    y = (table.match_frac >= 0.5).to_numpy() if len(table) else np.zeros(0, bool)
    if len(np.unique(y)) < 2:          # nothing to learn (no instances, or all of one kind)
        return None
    clf = HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.05,
                                         l2_regularization=1.0, random_state=seed)
    clf.fit(table[pp.FEATURES].to_numpy(np.float32), y)
    return clf


_NO_FILTER = dict(min_area=0, min_mean=0.0, clf=None)


def crossfit_chain(prob_dir, items, cache, folds, steps, postproc=pp.enhanced, n_jobs=4,
                   refit_all=True, log=print):
    """Greedy, cross-fitted post-processing chain.

    steps: list of dicts with keys name, grid (dict of lists), optional fixed (dict) and
    optional learn=True (train a fold-specific instance classifier first; its grid tunes clf_thr).
    Returns a list with one dict per step: params (context -> kwargs; contexts are the held-out
    folds 0..k-1 and "all"), oof (per-set DataFrame, each observation scored with the parameters
    tuned on the other folds), oof_summary, and grid (summary per config in the "all" context)."""
    k = max(folds.values()) + 1
    ctxs = list(range(k)) + (["all"] if refit_all else [])
    tune_obs = {c: [o for o in folds if (c == "all" or folds[o] != c)] for c in ctxs}
    base = {c: {} for c in ctxs}
    out = []
    for step in steps:
        t0 = time.time()
        fixed = step.get("fixed", {})
        extra = {c: {} for c in ctxs}
        if step.get("learn"):
            for c in ctxs:
                sub = [it for it in items if it["obs"] in set(tune_obs[c])]
                table = instance_table(prob_dir, sub, cache, {**base[c], **fixed, **_NO_FILTER}, postproc, n_jobs)
                extra[c]["clf"] = fit_filter(table)
        cfgs = {c: expand(step["grid"], {**base[c], **fixed, **extra[c]}) for c in ctxs}

        tune_sets = {c: set(v) for c, v in tune_obs.items()}

        def configs_for(o):
            return [((c, j), kw) for c in ctxs if o in tune_sets[c] for j, kw in enumerate(cfgs[c])]

        df = evaluate_configs(prob_dir, items, cache, configs_for, postproc, n_jobs)
        df["ctx"] = df.cid.map(lambda t: t[0])
        df["j"] = df.cid.map(lambda t: t[1])
        grid_tab = None
        for c in ctxs:
            d = df[df.ctx == c]
            score = d.groupby(["j", "obs"]).PQ.mean().groupby("j").mean()
            best = int(score.idxmax())           # first maximum: earlier (simpler) configs win ties
            base[c] = cfgs[c][best]
            if c == "all":
                grid_tab = pd.DataFrame([dict(_show(cfgs[c][j]), **summary(d[d.j == j])) for j in score.index])
                grid_tab = grid_tab.sort_values("PQ", ascending=False).reset_index(drop=True)
        # held-out evaluation: each observation scored with the chain tuned without its fold
        oof_s, oof = score_params(prob_dir, items, cache, lambda o: base[folds[o]], postproc, n_jobs)
        res = dict(name=step["name"], params={c: dict(base[c]) for c in ctxs}, oof=oof,
                   oof_summary=oof_s, grid=grid_tab, minutes=(time.time() - t0) / 60)
        out.append(res)
        log(f"{step['name']:<32} OOF PQ {oof_s['PQ']:.4f} (FP {oof_s['FP']}, FN {oof_s['FN']}, "
            f"split {oof_s.get('one_to_many')}, merge {oof_s.get('many_to_one')})  "
            f"chosen[all]: {_show(base.get('all', base[0]))}  [{res['minutes']:.1f} min]")
    return out


def _show(kw):
    "Parameters without the classifier object, for tables and logs."
    return {k: ("trained" if k == "clf" and v is not None else v) for k, v in kw.items()}


def apply_chain(prob_dir, items, cache, folds, step_result, postproc=pp.enhanced, n_jobs=4):
    "Score another set of probabilities (e.g. another seed) with a step's cross-fitted parameters."
    params = step_result["params"]
    return score_params(prob_dir, items, cache, lambda o: params[folds[o]], postproc, n_jobs)


def mean_std(summaries, keys=("PQ", "SQ", "RQ", "TP", "FP", "FN", "one_to_many", "many_to_one", "pp_s")):
    "Mean and std over seeds of the summary dicts."
    d = pd.DataFrame(summaries)
    out = {}
    for k in keys:
        if k in d:
            out[k] = d[k].mean()
            out[k + "_std"] = d[k].std(ddof=1) if len(d) > 1 else np.nan
    return out


def paired_bootstrap(df_a, df_b, n=5000, seed=0):
    "Change in per-observation PQ from a to b: mean and 95% bootstrap interval over observations."
    a, b = per_obs_pq(df_a), per_obs_pq(df_b)
    d = (b - a).dropna().to_numpy()
    rng = np.random.default_rng(seed)
    boots = d[rng.integers(0, len(d), (n, len(d)))].mean(1)
    return dict(delta=d.mean(), lo=np.percentile(boots, 2.5), hi=np.percentile(boots, 97.5),
                better=int((d > 0).sum()), worse=int((d < 0).sum()), n_obs=len(d))


# ----------------------------------------------------------------------------- timing

def benchmark_pipeline(predict_fn, paths, preprocess, postproc=pp.enhanced, pp_kwargs=None, warmup=2):
    """End-to-end seconds per image (raw file -> preprocess -> network -> instances), like
    evaluate.benchmark but for any predict_fn(img) (TTA, ensembles, pruned models, ...)."""
    times = []
    for i, p in enumerate(paths):
        t0 = time.perf_counter()
        img, disk = preprocess(p)
        t1 = time.perf_counter()
        probs = predict_fn(img)
        t2 = time.perf_counter()
        postproc(probs, disk, **(pp_kwargs or {}))
        t3 = time.perf_counter()
        if i >= warmup:
            times.append(dict(preprocess_s=t1 - t0, network_s=t2 - t1, postproc_s=t3 - t2, total_s=t3 - t0))
    return pd.DataFrame(times).agg(["mean", "std"])


# ----------------------------------------------------------------------------- figures

def _colour(lab, seed=0):
    rng = np.random.default_rng(seed)
    pal = np.vstack([[0, 0, 0], rng.uniform(0.25, 1, (int(lab.max()) + 1, 3))])
    return pal[lab]


def show_examples(prob_dir, cache, it, pipelines, path=None, down=4, crop=None):
    """One row per observation: image, first annotator's filaments, then each pipeline's
    instances (each instance a random colour), with that pipeline's PQ against the annotator."""
    import matplotlib.pyplot as plt
    probs = load_probs(prob_dir, it["obs"])
    img, disk = ev.load_obs(cache, it["obs"])
    gt = ev.load_gts(cache, it["image_ids"][:1])[0]
    sl = (slice(None), slice(None)) if crop is None else (slice(crop[0], crop[1]), slice(crop[2], crop[3]))
    d = 1 if crop is not None else down
    panels = [("image", img[sl][::d, ::d], None), ("annotation", _colour(gt[sl][::d, ::d]), None)]
    for name, (fn, kw) in pipelines.items():
        pred = fn(probs, disk, **kw)
        q = panoptic_quality(pred, gt)
        panels.append((f"{name}\nPQ {q['PQ']:.2f}  TP {q['TP']} FP {q['FP']} FN {q['FN']}",
                       _colour(pred[sl][::d, ::d]), None))
    fig, ax = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4.3))
    for a, (title, im, _) in zip(ax, panels):
        a.imshow(im, cmap="gray" if im.ndim == 2 else None)
        a.set_title(title, fontsize=9)
        a.axis("off")
    fig.suptitle(it["obs"], fontsize=10)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130, bbox_inches="tight")
    return fig


def plot_ablation(table, path=None, title="Cumulative post-processing ablation (out-of-fold, mean ± std over seeds)"):
    import matplotlib.pyplot as plt
    t = table.iloc[::-1]
    fig, ax = plt.subplots(1, 3, figsize=(15, 0.45 * len(t) + 1.5), sharey=True)
    ax[0].barh(t.step, t.PQ, xerr=t.get("PQ_std"), color="#2f6fb5", capsize=3)
    ax[0].set_xlabel("validation PQ")
    std = t.get("PQ_std", t.PQ * 0).fillna(0)
    for y, (v, s) in enumerate(zip(t.PQ, std)):
        ax[0].text(v + s + 0.003, y, f"{v:.3f}", va="center", fontsize=8)
    ax[0].set_xlim(max(0, t.PQ.min() - 0.05), (t.PQ + std).max() + 0.03)
    ax[1].barh(t.step, t.FP, color="#b8741a", label="false positives")
    ax[1].barh(t.step, -t.FN, color="#888", label="misses (left)")
    ax[1].set_xlabel("instances (seed mean)"); ax[1].legend(fontsize=8)
    ax[2].barh(t.step, t.one_to_many, color="#1f8a70", label="fragmented GT")
    ax[2].barh(t.step, -t.many_to_one, color="#a33", label="merged (left)")
    ax[2].set_xlabel("filaments (seed mean)"); ax[2].legend(fontsize=8)
    for a in ax:
        a.grid(alpha=0.3, axis="x")
    fig.suptitle(title)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130, bbox_inches="tight")
    return fig


def plot_pareto(df, path=None, x="total_s", y="PQ", label="pipeline"):
    "PQ against end-to-end seconds per image; the non-dominated pipelines are joined by a line."
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(df[x], df[y], yerr=df.get(y + "_std"), fmt="o", capsize=3)
    for i, (_, r) in enumerate(df.sort_values(x).iterrows()):
        ax.annotate(r[label], (r[x], r[y]), textcoords="offset points", xytext=(6, 10 if i % 2 else -12),
                    fontsize=8, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))
    front = df.sort_values(x)
    front = front[front[y] >= front[y].cummax()]
    ax.plot(front[x], front[y], "--", color="gray", alpha=0.6)
    ax.set(xscale="log", xlabel="end-to-end seconds per image (log)", ylabel="validation PQ")
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130, bbox_inches="tight")
    return fig


def plot_instance_errors(table, path=None):
    "Feature distributions of predicted instances that annotators accept (TP) or not (FP)."
    import matplotlib.pyplot as plt
    tp = table.match_frac >= 0.5
    cols = [("area", True), ("mean_fg", False), ("rdist", False), ("elongation", True),
            ("mean_spine", False), ("mean_width", True)]
    fig, ax = plt.subplots(2, 3, figsize=(14, 7))
    for a, (c, logx) in zip(ax.ravel(), cols):
        v = table[c].clip(lower=1e-3) if logx else table[c]
        bins = np.logspace(np.log10(v.min()), np.log10(v.max()), 40) if logx else 40
        a.hist(v[tp], bins=bins, alpha=0.6, label=f"matched ({tp.sum()})", color="#1f8a70")
        a.hist(v[~tp], bins=bins, alpha=0.6, label=f"false positive ({(~tp).sum()})", color="#b8741a")
        if logx:
            a.set_xscale("log")
        a.set_title(c); a.legend(fontsize=8)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130, bbox_inches="tight")
    return fig

In [ ]:
%%writefile src/compress.py
"""Model compression for the Segmentation Enhancement section (lecture: architecture selection
and compression, slides 16-24).

* unstructured magnitude pruning: zero the smallest-|w| convolution weights (global threshold).
  The tensors keep their shape, so the network is no faster on a GPU; the saved file only
  shrinks once it is compressed (we report gzip size to show this).
* structured pruning: remove whole convolution channels with the smallest L1 norm, using
  torch-pruning's dependency graph so skip connections and BatchNorm stay consistent. The model
  really gets smaller and faster, but needs fine-tuning afterwards (iterative pruning).
* knowledge distillation: a smaller student is trained on the ground truth and on the softened
  outputs of a teacher (here the ensemble of the three final seeds), per pixel and per channel.
"""
import gzip
import io
import json
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils import prune

from .unet import TARGET_CHANNELS


# ----------------------------------------------------------------------------- size

def count_params(model, nonzero=False):
    return sum(int((p != 0).sum()) if nonzero else p.numel() for p in model.parameters())


def stored_size_mb(model, half=True):
    """Size of the saved state_dict in MB, raw and gzip-compressed (fp16 if half). Zeros left by
    unstructured pruning only pay off in the compressed size."""
    sd = {k: (v.half() if half and v.is_floating_point() else v) for k, v in model.state_dict().items()}
    buf = io.BytesIO()
    torch.save(sd, buf)
    raw = buf.getvalue()
    return dict(file_MB=len(raw) / 2 ** 20, gzip_MB=len(gzip.compress(raw, 6)) / 2 ** 20)


def describe(model, example=None):
    "Parameters (total and non-zero), file sizes, and GFLOPs (MACs x 2) for one example input."
    d = dict(params_M=count_params(model) / 1e6, nonzero_params_M=count_params(model, True) / 1e6,
             **stored_size_mb(model))
    if example is not None:
        try:
            import torch_pruning as tp
            macs, _ = tp.utils.count_ops_and_params(model, example)
            d["GFLOPs"] = 2 * macs / 1e9
        except Exception as e:                      # counting is optional
            d["GFLOPs"] = float("nan")
            print("FLOP count failed:", e)
    return d


# ----------------------------------------------------------------------------- unstructured

def _prunable(model, skip=()):
    skip = set(id(m) for m in skip)
    return [(m, "weight") for m in model.modules()
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)) and id(m) not in skip]


def magnitude_prune(model, amount, skip=()):
    """Global unstructured L1 (magnitude) pruning of every conv weight except `skip` modules.
    The mask is folded into the weights (prune.remove), leaving plain zeros."""
    params = _prunable(model, skip)
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
    for m, n in params:
        prune.remove(m, n)
    return model


def conv_sparsity(model):
    ws = [m.weight for m, _ in _prunable(model)]
    return float(sum(int((w == 0).sum()) for w in ws) / sum(w.numel() for w in ws))


# ----------------------------------------------------------------------------- structured

def head_modules(model):
    "Layers whose output channels must not be pruned: the final 1x1 conv producing the 5 maps."
    if hasattr(model, "net"):                       # SMPUNet
        return [model.net.segmentation_head]
    return [model.head]


def structured_prune(model, ratio, example, round_to=None):
    """Remove `ratio` of the channels of every prunable layer, ranked by the L1 norm of each
    channel group (lecture: magnitude criterion). Returns the same model object, now smaller."""
    import torch_pruning as tp
    model.eval()
    imp = tp.importance.GroupMagnitudeImportance(p=1)
    pruner = tp.pruner.MetaPruner(model, example, importance=imp, pruning_ratio=ratio,
                                  ignored_layers=head_modules(model), round_to=round_to)
    pruner.step()
    return model


def save_full(model, path):
    "Pruned models no longer match build_model's architecture, so save the whole module."
    torch.save(model, path)


def load_full(path, device):
    return torch.load(path, map_location=device, weights_only=False).eval()


# ----------------------------------------------------------------------------- distillation

class DistillLoss(nn.Module):
    """alpha * (the normal multi-task loss on the annotations)
       + (1 - alpha) * per-channel distillation towards the teacher ensemble.

    Binary channels: BCE between the student's and the teacher's sigmoids at temperature T,
    scaled by T^2 (Hinton et al., 2015; a sigmoid is the 2-class softmax). The `dist` channel is a
    regression target, so it is matched with MSE at T=1. Pixel weights (zero off the disk) and
    channel weights are the same as in the normal loss."""

    def __init__(self, base_loss, teachers, alpha=0.5, T=2.0):
        super().__init__()
        self.base, self.alpha, self.T = base_loss, alpha, T
        self.teachers = teachers
        for t in teachers:
            t.eval()
            for p in t.parameters():
                p.requires_grad_(False)

    @torch.no_grad()
    def teacher_logits(self, x):
        with torch.autocast(x.device.type, dtype=torch.float16, enabled=x.device.type == "cuda"):
            return sum(t(x).float() for t in self.teachers) / len(self.teachers)

    def forward(self, logits, batch):
        hard, parts = self.base(logits, batch)
        tl = self.teacher_logits(batch["image"])
        w = batch["weight"]
        wsum = w.sum().clamp_min(1.0)
        kd = logits.new_zeros((), dtype=torch.float32)
        for c, name in enumerate(TARGET_CHANNELS):
            lam = self.base.weights.get(name)
            if not lam:
                continue
            z, t = logits[:, c:c + 1].float(), tl[:, c:c + 1]
            if name == "dist":
                term = (w * (torch.sigmoid(z) - torch.sigmoid(t)) ** 2).sum() / wsum
            else:
                soft = torch.sigmoid(t / self.T)
                term = (w * F.binary_cross_entropy_with_logits(z / self.T, soft, reduction="none")).sum() / wsum
                term = term * self.T ** 2
            kd = kd + lam * term
        parts = {**parts, "kd": kd.detach()}
        return self.alpha * hard + (1 - self.alpha) * kd, parts


# ----------------------------------------------------------------------------- bookkeeping

def write_json(path, d):
    Path(path).write_text(json.dumps(d, indent=1, default=float))


def timed(fn, *a, **kw):
    t0 = time.time()
    out = fn(*a, **kw)
    return out, time.time() - t0

In [ ]:
%%writefile src/submission.py
"""Write instance maps to a Kaggle submission CSV.

The exact column layout is set by the competition (see its Evaluation page and
sample_submission.csv), so it is a parameter here: `FORMAT` in notebook 03 says which column
holds the image id, whether there is one row per filament or one per image, and how a mask is
encoded. Check the first rows against the sample file before submitting.

Encoders:
* "rle"      Kaggle-style run-length string "start length start length ...", 1-indexed,
             pixels numbered row by row (order="C") or column by column (order="F")
* "coco_rle" pycocotools compressed RLE string (column-major), as stored in COCO files
"""
from pathlib import Path

import numpy as np
import pandas as pd


def rle_encode(mask, order="C"):
    px = np.concatenate([[0], np.asarray(mask, bool).ravel(order=order).astype(np.int8), [0]])
    runs = np.flatnonzero(px[1:] != px[:-1]) + 1
    runs[1::2] -= runs[::2]
    return " ".join(map(str, runs))


def rle_decode(s, shape, order="C"):
    v = np.asarray(str(s).split(), int) if isinstance(s, str) and s.strip() else np.zeros(0, int)
    m = np.zeros(shape[0] * shape[1], bool)
    for start, n in zip(v[0::2] - 1, v[1::2]):
        m[start:start + n] = True
    return m.reshape(shape, order=order)


def coco_rle_encode(mask):
    from pycocotools import mask as mu
    r = mu.encode(np.asfortranarray(np.asarray(mask, np.uint8)))
    return r["counts"].decode("ascii")


def encode(mask, encoding="rle", order="C"):
    return coco_rle_encode(mask) if encoding == "coco_rle" else rle_encode(mask, order)


def submission_rows(image_id, inst, fmt):
    """fmt keys: id_col, mask_col, per ("instance" or "image"), encoding, order, optional
    instance_col (1, 2, ... per image), optional extra (dict of constant columns),
    optional empty (value written for an image with no filaments)."""
    rows = []
    n = int(inst.max())
    if fmt.get("per", "instance") == "image":
        # one row per image; instances as separate RLE strings joined by fmt["sep"] or an id map
        enc = [encode(inst == k, fmt.get("encoding", "rle"), fmt.get("order", "C")) for k in range(1, n + 1)]
        rows.append({fmt["id_col"]: image_id, fmt["mask_col"]: fmt.get("sep", "|").join(enc) or fmt.get("empty", "")})
    else:
        for k in range(1, n + 1):
            r = {fmt["id_col"]: image_id, fmt["mask_col"]: encode(inst == k, fmt.get("encoding", "rle"), fmt.get("order", "C"))}
            if fmt.get("instance_col"):
                r[fmt["instance_col"]] = k
            rows.append(r)
        if n == 0 and "empty" in fmt:
            rows.append({fmt["id_col"]: image_id, fmt["mask_col"]: fmt["empty"]})
    for r in rows:
        r.update(fmt.get("extra", {}))
    return rows


def write_submission(pred_dir, out_csv, fmt, image_ids=None):
    """pred_dir holds <stem>.npz files with an `inst` array (notebooks 02/03). image_ids maps a
    file stem to the id Kaggle expects (default: the stem itself)."""
    rows = []
    for f in sorted(Path(pred_dir).glob("*.npz")):
        iid = image_ids.get(f.stem, f.stem) if image_ids else f.stem
        rows += submission_rows(iid, np.load(f)["inst"], fmt)
    cols = [fmt["id_col"]] + ([fmt["instance_col"]] if fmt.get("instance_col") else []) + [fmt["mask_col"]]
    df = pd.DataFrame(rows)
    df = df[cols + [c for c in df.columns if c not in cols]]
    if fmt.get("instance_col"):
        df[fmt["instance_col"]] = df[fmt["instance_col"]].astype("Int64")
    df.to_csv(out_csv, index=False)
    return df


def check_against_sample(sub, sample):
    "Same columns, and every image in the sample appears in the submission."
    problems = []
    if list(sub.columns) != list(sample.columns):
        problems.append(f"columns differ: submission {list(sub.columns)} vs sample {list(sample.columns)}")
    id_col = sample.columns[0]
    if id_col in sub:
        missing = set(sample[id_col].astype(str)) - set(sub[id_col].astype(str))
        if missing:
            problems.append(f"{len(missing)} sample ids missing, e.g. {sorted(missing)[:3]}")
    return problems

In [ ]:
# ---- settings ----------------------------------------------------------------------------------
RUN_NAME  = "C_final"     # notebook 02's final configuration (use "D_final" if it finishes and scores higher)
SEEDS     = (0, 1, 2)     # its three training seeds
TUNE_SEED = 1             # its best seed: post-processing is tuned on this seed's outputs, then checked on all three
N_JOBS    = os.cpu_count()
TRAIN_IF_MISSING = True   # no checkpoints attached: retrain the final model here with notebook 02's exact
RETRAIN_SEEDS = (1,)      # config files and training code, for these seeds only (about 75 min each on a T4).
                          # Seeds that are neither attached nor retrained are left out (no seed ensemble/std).
BASELINE = dict(thr=0.5, min_area=50)   # notebook 02's post-processing

DO_TTA, DO_HALF_RES, DO_FP32 = True, True, True
DO_UNSTRUCTURED, UNSTRUCTURED_AMOUNTS = True, (0.3, 0.5, 0.7, 0.9)
DO_STRUCTURED, STRUCTURED_STEPS = True, (0.25, 0.5)   # share of channels removed after each iterative step
FT_EPOCHS, FT_LR = 8, 3e-4                            # fine-tuning after each structured pruning step
DO_DISTILL = True
DISTILL = dict(encoder="resnet18", encoder_weights="imagenet", epochs=30, alpha=0.5, T=2.0, lr=3e-4)
DO_STUDENT_CONTROL = True     # the same student without the teacher, to isolate what distillation adds
RETUNE_BEST_VARIANT = True    # re-run the post-processing chain on the best probability variant (CPU only)
FAIR_RETUNE = True            # section 6b: tune the post-processing separately for every probability source and
                              # compressed model (CPU only), so each is compared at its own best settings
N_BENCH = 22                  # test images timed end to end (the first 2 are warm-up), as in notebook 02
MAX_HOURS = 10.5 if ON_KAGGLE else None   # Kaggle stops sessions at 12 h; no new training starts after this
DEADLINE = T_START + MAX_HOURS * 3600 if MAX_HOURS else None

def time_left():
    return DEADLINE is None or time.time() < DEADLINE

In [ ]:
# Where results go. On Kaggle /kaggle/working is kept as the version's output; results of an earlier
# version attached with "+ Add Input" are copied back in, so finished work is skipped.
if ON_KAGGLE:
    OUT, PROBS = Path("/kaggle/working/enhancement"), Path("/kaggle/temp/probs")
    for prev in Path("/kaggle/input").glob("**/enhancement/tables"):
        shutil.copytree(prev.parent, OUT, dirs_exist_ok=True)
        print("restored earlier results from", prev.parent)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    TEAM_DIR = next(d for d in [Path("/content/drive/Shareddrives/COS 711/Assignment 2"),
                                Path("/content/drive/MyDrive/COS 711/Assignment 2"),
                                Path("/content/drive/MyDrive/Assignment 2")] if d.exists())
    OUT, PROBS = TEAM_DIR / "enhancement", Path("/content/probs")
TAB, FIG, MODEL_DIR = OUT / "tables", OUT / "figures", OUT / "models"
for d in (TAB, FIG, MODEL_DIR, PROBS):
    d.mkdir(parents=True, exist_ok=True)
print("results:", OUT)

In [ ]:
# Notebook 01 finds the data and builds the 2048x2048 cache (this takes a while on a fresh session).
# It defines obs_meta, set_meta, FilamentDS, panoptic_quality, split_merge, fit_disk, normalise,
# circle_mask, DISK_MARGIN, CACHE, DATA.
from IPython.utils.capture import capture_output
if "set_meta" not in globals():
    try:
        with capture_output() as prep_log:
            get_ipython().run_line_magic("run", f"{REPO}/notebooks/01_data_preprocessing.ipynb")
    except Exception:
        prep_log.show()
        raise
print(f"cache: {CACHE}, {shutil.disk_usage(CACHE).free / 2**30:.0f} GB still free")

In [ ]:
import joblib
import numpy as np, pandas as pd, torch, cv2
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.unet import build_model, model_size
from src.losses import MultiTaskLoss
from src.train import fit, seed_everything
from src import evaluate as ev, postprocess as pp, inference as inf, enh_eval as ee, compress as cp, submission as sub
from src import metrics as M

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
CACHE_DIR = CACHE / "scale1.0"
VAL = ev.val_items(set_meta)
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "", "|", len(VAL), "validation observations,",
      sum(len(v["image_ids"]) for v in VAL), "annotation sets")

# src.metrics must match notebook 01's metric exactly (it is a copy, used by worker processes)
_g = [np.load(CACHE_DIR / "inst" / f"{i}.npy") for i in VAL[0]["image_ids"][:1] + VAL[1]["image_ids"][:1]]
assert M.panoptic_quality(_g[0], _g[1]) == panoptic_quality(_g[0], _g[1])
assert M.split_merge(_g[0], _g[1]) == split_merge(_g[0], _g[1])

In [ ]:
# Training helpers (same data pipeline, loss and training loop as notebook 02), used to retrain the final
# model if its checkpoints are not attached, and for fine-tuning after pruning and for distillation.
def make_train_dl(cfg):
    ds = FilamentDS(set_meta[set_meta.split == "train"], obs_meta, CACHE_DIR, crop=cfg["crop"], train=True, pos_prob=cfg["pos_prob"])
    nw = min(cfg["num_workers"], os.cpu_count())
    return DataLoader(ds, cfg["batch_size"], shuffle=True, drop_last=True, pin_memory=True,
                      num_workers=nw, persistent_workers=nw > 0)

def base_loss(cfg):
    return MultiTaskLoss(cfg["loss_weights"], cfg["pos_weight"], cfg["dice"], cfg["soft_label_alpha"])

val_fn = lambda m: ev.evaluate(m, VAL, CACHE_DIR, M.panoptic_quality, pp_kwargs=BASELINE)[0]

def train(model, cfg, run_name, loss_fn):
    "Train with notebook 02's fit(); returns True if all epochs finished (else it resumes next session)."
    run_dir = OUT / "runs" / run_name
    seed_everything(cfg["seed"])
    hist = fit(model, make_train_dl(cfg), loss_fn, cfg, run_dir, val_fn, deadline=DEADLINE)
    if len(hist) < cfg["epochs"]:
        print(f"{run_name}: stopped after {len(hist)} epochs (time budget); rerun to resume")
        return False
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
    (run_dir / "last.pt").unlink(missing_ok=True)
    return True

def train_final(seed):
    # Retrain notebook 02's final configuration for one seed: the same config.json (from the repo's
    # results/runs), the same FilamentDS, loss and fit(), and the same best-validation-PQ selection.
    name = f"{RUN_NAME}_s{seed}"
    cfg = json.loads((Path(REPO) / "results/runs" / name / "config.json").read_text())
    run_dir = OUT / "base_runs" / name
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(cfg, indent=1))
    print(f"=== training {name} here (notebook 02's settings): {cfg['epochs']} epochs")
    model = build_model(cfg).to(DEVICE)                     # downloads the ImageNet encoder weights
    seed_everything(cfg["seed"])
    t0 = time.time()
    hist = fit(model, make_train_dl(cfg), base_loss(cfg), cfg, run_dir,
               lambda m: ev.evaluate(m, VAL, CACHE_DIR, M.panoptic_quality, pp_kwargs=cfg["pp"])[0], deadline=DEADLINE)
    if len(hist) < cfg["epochs"]:
        raise RuntimeError(f"{name} stopped after {len(hist)} epochs (time budget). Save a new version with this "
                           "version's output attached (+ Add Input -> this notebook) and it resumes.")
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
    summ, per_set = ev.evaluate(model, VAL, CACHE_DIR, M.panoptic_quality, M.split_merge, pp_kwargs=cfg["pp"])
    per_set.to_csv(run_dir / "val_per_set.csv", index=False)
    (run_dir / "summary.json").write_text(json.dumps(dict(
        name=RUN_NAME, seed=seed, retrained_in_notebook_03=True, **model_size(model), epochs=cfg["epochs"],
        best_epoch=int(hist.loc[hist.val_PQ.idxmax(), "epoch"]), train_min=hist.train_s.sum() / 60,
        wall_min=(time.time() - t0) / 60, **{f"val_{k}": v for k, v in summ.items()}), indent=1, default=float))
    (run_dir / "last.pt").unlink(missing_ok=True)
    return run_dir

In [ ]:
# The final model's checkpoints (best.pt, one per seed) from notebook 02's output.
def find_run_dir(run, seed):
    name = f"{run}_s{seed}"
    roots = ([Path("/kaggle/input"), Path("/kaggle/working/runs")] if ON_KAGGLE else [TEAM_DIR / "runs"]) + [OUT / "base_runs"]
    for root in roots:
        hits = sorted(root.glob(f"**/{name}/best.pt")) if root.exists() else []
        # a run that is still training has last.pt next to best.pt (and no summary): not finished
        hits = [h for h in hits if (h.parent / "summary.json").exists() or not (h.parent / "last.pt").exists()]
        if hits:
            return hits[0].parent
    return None

CFG, MODELS, REF = None, {}, {}
for s in SEEDS:
    d = find_run_dir(RUN_NAME, s)
    if d is None and TRAIN_IF_MISSING and s in RETRAIN_SEEDS:
        d = train_final(s)
    if d is None and TRAIN_IF_MISSING:
        print(f"seed {s}: not attached and not in RETRAIN_SEEDS, left out")
        continue
    if d is None:
        raise FileNotFoundError(
            f"{RUN_NAME}_s{s}/best.pt not found. Add notebook 02's output (or a dataset holding the runs/ folder) "
            "with '+ Add Input' in the right-hand panel; see docs/kaggle_guide.md.")
    cfg_file = d / "config.json" if (d / "config.json").exists() else Path(REPO) / "results/runs" / d.name / "config.json"
    cfg = json.loads(cfg_file.read_text())
    m = build_model({**cfg, "encoder_weights": None}).to(DEVICE)
    m.load_state_dict(torch.load(d / "best.pt", map_location=DEVICE))
    MODELS[s] = m.eval()
    CFG = CFG or cfg
    ref = Path(REPO) / "results/runs" / d.name / "summary.json"          # what notebook 02 reported
    REF[s] = json.loads(ref.read_text())["val_PQ"] if ref.exists() else float("nan")
    own = json.loads((d / "summary.json").read_text()) if (d / "summary.json").exists() else {}
    note = (f", retrained here: {own['val_PQ']:.4f} after {own['train_min']:.0f} min"
            if own.get("retrained_in_notebook_03") else "")
    print(f"seed {s}: {d}  (notebook 02 validation PQ {REF[s]:.4f}{note})")
SEEDS = tuple(sorted(MODELS))
assert TUNE_SEED in MODELS, f"TUNE_SEED {TUNE_SEED} has no model: attach it or add it to RETRAIN_SEEDS"
print("seeds used:", SEEDS, {k: round(v, 2) for k, v in model_size(MODELS[TUNE_SEED]).items()})

In [ ]:
# Same preprocessing as notebook 02 (for raw test images), and the end-to-end timing helper.
def preprocess_raw(path):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    H0, W0 = gray.shape
    p = fit_disk(gray)
    cx, cy, r = p if p is not None else (W0 / 2, H0 / 2, min(H0, W0) / 2)
    disk = circle_mask(H0, W0, cx, cy, r * DISK_MARGIN)
    img8, _, _ = normalise(gray, disk)
    return img8.astype(np.float32) / 255, disk

TEST_PATHS = sorted((DATA / "test/test_images").glob("*.jp*g"))
TIMING_FILE = TAB / "timings.json"
TIMINGS = json.loads(TIMING_FILE.read_text()) if TIMING_FILE.exists() else {}

def bench(label, predict_fn, pp_kwargs):
    "End-to-end seconds per test image (raw JPEG -> disk fit -> network -> instances), cached by label."
    if label not in TIMINGS:
        t = ee.benchmark_pipeline(predict_fn, TEST_PATHS[:N_BENCH], preprocess_raw, pp.enhanced, pp_kwargs)
        TIMINGS[label] = {k: float(t.loc["mean", k]) for k in t.columns}
        TIMING_FILE.write_text(json.dumps(TIMINGS, indent=1))
    return TIMINGS[label]

def cache_variant(name, predict_fn):
    "Cache the validation probabilities of one prediction variant (skipped if already cached)."
    t = inf.cache_probs(predict_fn, VAL, CACHE_DIR, PROBS / name, progress=lambda x: tqdm(x, desc=name, leave=False))
    return PROBS / name

## 2. Cache the validation probabilities and reproduce the baseline

The network runs once per variant over the 107 validation disks; the probabilities are stored as 8-bit maps (`/kaggle/temp/probs`), so every post-processing experiment afterwards runs on CPU. The first check: notebook 02's baseline post-processing (fg threshold 0.5, minimum area 50 px) on these cached maps should give notebook 02's validation PQ (tiny differences come from storing probabilities with 8 bits). If the seeds were retrained in this notebook (no checkpoints attached), expect differences of about ±0.01 instead: the data loader's random crops are not seeded, so a retrained seed is a new sample of the same recipe, and notebook 02 measured the seed-to-seed spread as ±0.008.

In [ ]:
for s in SEEDS:
    cache_variant(f"s{s}", lambda img, m=MODELS[s]: inf.predict_probs(m, img))

for s in SEEDS:
    summ, _ = ee.score_params(PROBS / f"s{s}", VAL, CACHE_DIR, BASELINE, n_jobs=N_JOBS)
    print(f"seed {s}: notebook 02 {REF[s]:.4f}   here {summ['PQ']:.4f}   "
          f"(TP {summ['TP']}, FP {summ['FP']}, FN {summ['FN']}, split {summ['one_to_many']}, merge {summ['many_to_one']})")

In [ ]:
# The other probability variants (sections 5 and 6). Ensembles are averaged on CPU from the seeds' maps.
if DO_TTA:
    for s in SEEDS:
        cache_variant(f"s{s}_tta8", lambda img, m=MODELS[s]: inf.predict_probs(m, img, tta=8))
if DO_HALF_RES:
    for s in SEEDS:
        cache_variant(f"s{s}_half", lambda img, m=MODELS[s]: inf.predict_probs(m, img, scale=0.5))
if DO_FP32:
    cache_variant(f"s{TUNE_SEED}_fp32", lambda img: inf.predict_probs(MODELS[TUNE_SEED], img, amp=False))
ENSEMBLE = len(SEEDS) > 1
if ENSEMBLE:
    inf.average_cached([PROBS / f"s{s}" for s in SEEDS], PROBS / "ensemble", VAL)
if ENSEMBLE and DO_TTA:
    inf.average_cached([PROBS / f"s{s}_tta8" for s in SEEDS], PROBS / "ensemble_tta8", VAL)
print(pd.Series({d.name: json.loads((d / "timing.json").read_text()).get("network_s") for d in sorted(PROBS.iterdir())
                 if (d / "timing.json").exists()}, name="network s / disk (validation)").round(3))
print(f"probability maps: {sum(f.stat().st_size for f in PROBS.rglob('*.npz')) / 2**30:.1f} GB, "
      f"{shutil.disk_usage(PROBS).free / 2**30:.0f} GB disk still free")

## 3. What goes wrong with the baseline

Every predicted instance of the tuning seed is matched against every annotator. An instance counts as a true positive if it matches (IoU > 0.5) a filament for at least half of the annotators. The tables show where the false positives are: by size, by confidence and by position on the disk (`rdist` = distance from the disk centre in disk radii).

In [ ]:
inst_file = TAB / "baseline_instances.csv"
if inst_file.exists():
    inst_base = pd.read_csv(inst_file)
else:
    inst_base = ee.instance_table(PROBS / f"s{TUNE_SEED}", VAL, CACHE_DIR, BASELINE, n_jobs=N_JOBS)
    inst_base.to_csv(inst_file, index=False)
inst_base["FP"] = inst_base.match_frac < 0.5
print(f"{len(inst_base)} predicted instances, {inst_base.FP.mean():.0%} are false positives")

def fp_by(col, bins):
    g = inst_base.groupby(pd.cut(inst_base[col], bins), observed=True).FP
    return pd.DataFrame(dict(instances=g.size(), false_pos=g.sum(), fp_rate=g.mean().round(2)))

display(fp_by("area", [0, 100, 200, 500, 1000, 3000, 1e9]))
display(fp_by("mean_fg", [0, 0.6, 0.7, 0.8, 0.9, 1.0]))
display(fp_by("rdist", [0, 0.8, 0.9, 0.95, 1.05]))
ee.plot_instance_errors(inst_base, FIG / "baseline_instance_errors.png");

## 4. Instance extraction using all five output maps (cross-fitted ablation)

A greedy chain on the tuning seed's probabilities. Each step adds one stage and tunes only its own parameters, with the earlier stages fixed to what that fold's chain chose. Every grid includes the "do nothing" setting (listed first, which also wins ties), so a stage that does not help is switched off. `keep` means "leave this parameter as the earlier step chose it".

| Step | Stage | Parameters |
|---|---|---|
| baseline | threshold 0.5, connected components, min area 50 (notebook 02) | none |
| tuned threshold | notebook 02's grid, extended to larger areas | `thr`, `min_area` |
| hysteresis | low-confidence pixels kept only when connected to confident ones | `thr` (high), `lo`, `min_seed` |
| join fragments | pieces bridged by low fg, the spine map, or a small closing share a label | `join_r`, `join_thr`, `spine_thr` |
| split merges | watershed between parts separated by contact / boundary / distance-map valleys | `split`, `cut_thr`, `d_thr`, `min_part` |
| rule filter | drop small, low-confidence or limb instances | `min_area`, `min_mean`, `max_rdist` |
| learned filter | gradient-boosted classifier on 14 instance features, trained on the tuning fold | `clf_thr` |

In [ ]:
FOLDS = ee.make_folds(VAL, obs_meta, k=2)
print("observations per fold:", np.bincount(list(FOLDS.values())))
K = ee.KEEP
STEPS = [
    dict(name="baseline (nb 02)", grid={}, fixed=BASELINE),
    dict(name="tuned threshold + min area", grid=dict(thr=[K, 0.3, 0.4, 0.6, 0.7], min_area=[K, 0, 100, 200, 300, 500, 800])),
    dict(name="+ hysteresis", grid=dict(lo=[None, 0.2, 0.3, 0.4], thr=[K, 0.5, 0.6, 0.7, 0.8], min_seed=[0, 100])),
    dict(name="+ join fragments", grid=dict(join_r=[0, 2, 4, 8], join_thr=[None, 0.2], spine_thr=[None, 0.3, 0.5])),
    dict(name="+ split merges", grid=[dict(split=None)]
         + [dict(split=s, cut_thr=c, min_part=m) for s in ("contact", "boundary") for c in (0.3, 0.5) for m in (50, 150)]
         + [dict(split="dist", d_thr=d, min_part=m) for d in (0.3, 0.5) for m in (50, 150)]),
    dict(name="+ rule filter", grid=dict(min_area=[K, 50, 100, 200, 300, 500], min_mean=[0.0, 0.5, 0.6, 0.7], max_rdist=[None, 0.95])),
    dict(name="+ learned filter", learn=True, grid=[dict(clf_thr=0)]
         + [dict(clf_thr=t, min_area=a, min_mean=0.0) for t in (0.3, 0.4, 0.5, 0.6) for a in (0, 50, 100)]),
]

def run_chain(prob_dir, tag, steps=None):
    f = OUT / f"chain_{tag}.pkl"
    if f.exists():
        return joblib.load(f)
    chain = ee.crossfit_chain(prob_dir, VAL, CACHE_DIR, FOLDS, steps or STEPS, n_jobs=N_JOBS)
    joblib.dump(chain, f)
    return chain

CHAIN = run_chain(PROBS / f"s{TUNE_SEED}", f"s{TUNE_SEED}")
print("\nIn-sample grid of the tuned-threshold step (notebook 02 reported 0.405 for thr 0.4, min area 200):")
display(CHAIN[1]["grid"].head(5).round(4))

In [ ]:
# Every step scored out-of-fold on all three seeds (parameters tuned on the tuning seed).
def obs_mean_pq(dfs):
    "Per-observation PQ averaged over seeds."
    return pd.concat([ee.per_obs_pq(d) for d in dfs], axis=1).mean(1)

def boot(a, b, n=5000, seed=0):
    d = (b - a).dropna().to_numpy()
    rng = np.random.default_rng(seed)
    bs = d[rng.integers(0, len(d), (n, len(d)))].mean(1)
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)

def chain_table(chain, dirs):
    rows, per_obs = [], []
    for r in chain:
        res = [ee.apply_chain(d, VAL, CACHE_DIR, FOLDS, r, n_jobs=N_JOBS) for d in dirs]
        rows.append(dict(step=r["name"], **ee.mean_std([x[0] for x in res])))
        per_obs.append(obs_mean_pq([x[1] for x in res]))
    tab = pd.DataFrame(rows)
    for col in ("vs_prev", "vs_baseline"):
        tab[f"dPQ_{col}"], tab[f"dPQ_{col}_CI"] = np.nan, ""
    for i in range(1, len(tab)):
        for ref, col in ((i - 1, "vs_prev"), (0, "vs_baseline")):
            m, lo, hi = boot(per_obs[ref], per_obs[i])
            tab.loc[i, f"dPQ_{col}"] = m
            tab.loc[i, f"dPQ_{col}_CI"] = f"[{lo:+.3f}, {hi:+.3f}]"
    return tab

ABL_FILE = TAB / "ablation_postprocessing.csv"
if ABL_FILE.exists():
    ABL = pd.read_csv(ABL_FILE)
else:
    ABL = chain_table(CHAIN, [PROBS / f"s{s}" for s in SEEDS])
    ABL.to_csv(ABL_FILE, index=False)
cols = ["step", "PQ", "PQ_std", "dPQ_vs_prev", "dPQ_vs_prev_CI", "dPQ_vs_baseline", "SQ", "RQ", "TP", "FP", "FN",
        "one_to_many", "many_to_one", "pp_s"]
display(ABL[[c for c in cols if c in ABL]].round(4))
ee.plot_ablation(ABL, FIG / "ablation_postprocessing.png");

params = {r["name"]: {c: ee._show(p) for c, p in r["params"].items()} for r in CHAIN}
(TAB / "chain_params.json").write_text(json.dumps(params, indent=1, default=str))
print("final chain parameters (tuned on all validation data, used for the test set):")
print(ee._show(CHAIN[-1]["params"]["all"]))

In [ ]:
# Examples for the slides: the observations with the largest gain and the largest loss, and a zoom on
# the filament the baseline fragments most.
FINAL_PARAMS = CHAIN[-1]["params"]["all"]
gain = (ee.per_obs_pq(CHAIN[-1]["oof"]) - ee.per_obs_pq(CHAIN[0]["oof"])).sort_values()
by_obs = {it["obs"]: it for it in VAL}
pipes = {"baseline (nb 02)": (pp.enhanced, BASELINE), "enhanced": (pp.enhanced, FINAL_PARAMS)}
for tag, o in [("best_gain", gain.index[-1]), ("worst_change", gain.index[0]), ("median", gain.index[len(gain) // 2])]:
    ee.show_examples(PROBS / f"s{TUNE_SEED}", CACHE_DIR, by_obs[o], pipes, FIG / f"example_{tag}.png")
    plt.show()

def zoom_box(obs, size=640):
    probs = inf.load_probs(PROBS / f"s{TUNE_SEED}", obs)
    _, disk = ev.load_obs(CACHE_DIR, obs)
    pred = pp.enhanced(probs, disk, **BASELINE)
    gt = ev.load_gts(CACHE_DIR, by_obs[obs]["image_ids"][:1])[0]
    best, centre = 0, None
    for k in range(1, int(gt.max()) + 1):
        ys, xs = np.nonzero(gt == k)
        if len(ys) == 0:
            continue
        n = len(np.unique(pred[ys, xs])) - (pred[ys, xs] == 0).any()
        if n > best:
            best, centre = n, (int(ys.mean()), int(xs.mean()))
    if centre is None:
        return None
    size = min(size, *gt.shape)
    y0 = int(np.clip(centre[0] - size // 2, 0, gt.shape[0] - size)); x0 = int(np.clip(centre[1] - size // 2, 0, gt.shape[1] - size))
    return (y0, y0 + size, x0, x0 + size)

o = gain.index[-1]
box = zoom_box(o)
if box:
    ee.show_examples(PROBS / f"s{TUNE_SEED}", CACHE_DIR, by_obs[o], pipes, FIG / "example_zoom.png", crop=box)

## 5. Better probabilities: test-time augmentation and the seed ensemble

The final post-processing chain (cross-fitted, so still out-of-fold) applied to other probability maps: the 8 rotations/reflections of each disk averaged (TTA ×8), the three seeds averaged (ensemble), and both. TTA ×8 costs 8 network passes, the ensemble 3, and both 24. The ensemble is a single model set, so it has no seed spread.

In [ ]:
FINAL_STEP = CHAIN[-1]

def score_dirs(dirs, step=FINAL_STEP):
    res = [ee.apply_chain(d, VAL, CACHE_DIR, FOLDS, step, n_jobs=N_JOBS)[0] for d in dirs]
    return ee.mean_std(res)

def network_s(name):
    f = PROBS / name / "timing.json"
    return json.loads(f.read_text()).get("network_s") if f.exists() else np.nan

VAR_FILE = TAB / "probability_variants.csv"
if VAR_FILE.exists():
    VARS = pd.read_csv(VAR_FILE)
else:
    rows = [dict(variant="single model", passes=1, **score_dirs([PROBS / f"s{s}" for s in SEEDS]),
                 val_network_s=network_s(f"s{TUNE_SEED}"))]
    if DO_TTA:
        rows.append(dict(variant="TTA x8", passes=8, **score_dirs([PROBS / f"s{s}_tta8" for s in SEEDS]),
                         val_network_s=network_s(f"s{TUNE_SEED}_tta8")))
    if ENSEMBLE:
        rows.append(dict(variant="3-seed ensemble", passes=3, **score_dirs([PROBS / "ensemble"]), val_network_s=network_s("ensemble")))
    if ENSEMBLE and DO_TTA:
        rows.append(dict(variant="ensemble + TTA x8", passes=24, **score_dirs([PROBS / "ensemble_tta8"]),
                         val_network_s=network_s("ensemble_tta8")))
    VARS = pd.DataFrame(rows)
    VARS.to_csv(VAR_FILE, index=False)
display(VARS[["variant", "passes", "PQ", "PQ_std", "SQ", "RQ", "FP", "FN", "one_to_many", "many_to_one", "val_network_s"]].round(4))

In [ ]:
# The chain was tuned on single-model probabilities. Averaged maps are smoother, so the best variant
# may prefer other thresholds: re-run the whole cross-fitted chain on it.
VARIANT_DIRS = {"single model": f"s{TUNE_SEED}", "TTA x8": f"s{TUNE_SEED}_tta8",
                "3-seed ensemble": "ensemble", "ensemble + TTA x8": "ensemble_tta8"}
best_variant = VARS.sort_values("PQ", ascending=False).variant.iloc[0]
CHAINS = {"single model": CHAIN}
if RETUNE_BEST_VARIANT and best_variant != "single model":
    CHAINS[best_variant] = run_chain(PROBS / VARIANT_DIRS[best_variant], VARIANT_DIRS[best_variant])
    retuned = CHAINS[best_variant][-1]["oof_summary"]
    print(f"{best_variant}: OOF PQ {VARS.set_index('variant').PQ[best_variant]:.4f} with the single-model chain, "
          f"{retuned['PQ']:.4f} with its own chain")
    pd.DataFrame([dict(step=r["name"], **r["oof_summary"]) for r in CHAINS[best_variant]]).to_csv(
        TAB / f"ablation_{VARIANT_DIRS[best_variant]}.csv", index=False)

## 6. Compression

Accuracy against cost, all with the final post-processing chain (out-of-fold PQ on the tuning seed). Times are end to end on test images (raw JPEG → disk fit → network → instances), measured the same way as notebook 02's 0.40 s.

* **Precision and resolution:** fp32 vs fp16 autocast (notebook 02 already used fp16), and running the network on a half-resolution disk (4× fewer pixels).
* **Unstructured magnitude pruning** (lecture slides 16-17): the smallest-magnitude 30-90% of convolution weights are set to zero, no retraining. The tensors keep their shape, so it saves nothing at run time; only the compressed file shrinks.
* **Iterative structured pruning** (slides 17-18): remove 25% of every layer's channels (smallest L1 norm), fine-tune, then prune to 50% and fine-tune again. This really shrinks the network.
* **Knowledge distillation** (slides 21-23): a ResNet-18 U-Net student learns from the annotations and from the 3-seed ensemble's softened outputs (T = 2, α = 0.5). The control is the same student trained on the annotations only.

In [ ]:
COMP_FILE = TAB / "compression.csv"
COMP = pd.read_csv(COMP_FILE).to_dict("records") if COMP_FILE.exists() else []
done = {r["model"] for r in COMP}
EX = torch.randn(1, 1, 1024, 1024, device=DEVICE)          # FLOPs are counted at 1024^2 and scaled x4 to a full disk
TUNE_STEP = CHAINS["single model"][-1]

def add_row(label, model, prob_name, predict_fn, note="", **extra):
    if label in done:
        return
    s, _ = ee.apply_chain(PROBS / prob_name, VAL, CACHE_DIR, FOLDS, TUNE_STEP, n_jobs=N_JOBS)
    d = cp.describe(model, EX) if model is not None else {}
    if "GFLOPs" in d:
        d["GFLOPs_2048"] = d.pop("GFLOPs") * 4
    t = bench(label, predict_fn, TUNE_STEP["params"]["all"])
    COMP.append(dict(model=label, PQ=s["PQ"], SQ=s["SQ"], RQ=s["RQ"], FP=s["FP"], FN=s["FN"], **d,
                     network_s=t["network_s"], total_s=t["total_s"], note=note, **extra))
    pd.DataFrame(COMP).to_csv(COMP_FILE, index=False)
    done.add(label)

base_m = MODELS[TUNE_SEED]
add_row("ResNet-34 U-Net, fp16 (nb 02)", base_m, f"s{TUNE_SEED}", lambda img: inf.predict_probs(base_m, img))
if DO_FP32:
    add_row("ResNet-34 U-Net, fp32", base_m, f"s{TUNE_SEED}_fp32", lambda img: inf.predict_probs(base_m, img, amp=False))
if DO_HALF_RES:
    add_row("ResNet-34 U-Net, half resolution", base_m, f"s{TUNE_SEED}_half", lambda img: inf.predict_probs(base_m, img, scale=0.5))
pd.DataFrame(COMP).round(4)

In [ ]:
# Unstructured magnitude pruning, no fine-tuning.
if DO_UNSTRUCTURED:
    for a in UNSTRUCTURED_AMOUNTS:
        label = f"unstructured {a:.0%} zeros"
        if label in done:
            continue
        m = cp.magnitude_prune(copy.deepcopy(base_m), a, skip=cp.head_modules(base_m)).eval()
        name = f"s{TUNE_SEED}_unstr{int(a * 100)}"
        cache_variant(name, lambda img, m=m: inf.predict_probs(m, img))
        add_row(label, m, name, lambda img, m=m: inf.predict_probs(m, img), sparsity=cp.conv_sparsity(m))
        shutil.rmtree(PROBS / name, ignore_errors=True)        # not needed again; saves disk space
        del m; torch.cuda.empty_cache()
pd.DataFrame(COMP).round(4)

In [ ]:
# Iterative structured pruning with fine-tuning.
if DO_STRUCTURED:
    cur, removed = copy.deepcopy(base_m), 0.0
    ex_small = torch.randn(1, 1, 512, 512, device=DEVICE)
    for target in STRUCTURED_STEPS:
        name = f"structured{int(target * 100)}"
        f = MODEL_DIR / f"{name}.pt"
        if f.exists():
            cur = cp.load_full(f, DEVICE)
        else:
            cp.structured_prune(cur, 1 - (1 - target) / (1 - removed), ex_small)   # share of the remaining channels
            cache_variant(f"{name}_noft", lambda img, m=cur: inf.predict_probs(m, img))
            add_row(f"structured {target:.0%} channels removed, before fine-tuning", cur, f"{name}_noft",
                    lambda img, m=cur: inf.predict_probs(m, img))
            shutil.rmtree(PROBS / f"{name}_noft", ignore_errors=True)
            ft_cfg = {**CFG, "epochs": FT_EPOCHS, "lr": FT_LR, "warmup_epochs": 1, "val_every": FT_EPOCHS}
            if not (time_left() and train(cur.train(), ft_cfg, name, base_loss(CFG))):
                break
            cp.save_full(cur.eval(), f)
        removed = target
        cache_variant(name, lambda img, m=cur: inf.predict_probs(m, img))
        add_row(f"structured {target:.0%} channels removed + {FT_EPOCHS} ep fine-tune", cur, name,
                lambda img, m=cur: inf.predict_probs(m, img))
pd.DataFrame(COMP).round(4)

In [ ]:
# Knowledge distillation: the 3-seed ensemble teaches a ResNet-18 U-Net; the control has no teacher.
if DO_DISTILL:
    s_cfg = {**CFG, "encoder": DISTILL["encoder"], "encoder_weights": DISTILL["encoder_weights"], "epochs": DISTILL["epochs"],
             "lr": DISTILL["lr"], "val_every": 5, "seed": 0}
    runs = [("student_kd", True)] + ([("student_plain", False)] if DO_STUDENT_CONTROL else [])
    for name, use_kd in runs:
        f = MODEL_DIR / f"{name}.pt"
        if f.exists():
            student = cp.load_full(f, DEVICE)
        else:
            if not time_left():
                print("time budget used up, not starting", name); continue
            student = build_model(s_cfg).to(DEVICE)
            loss = (cp.DistillLoss(base_loss(CFG), [MODELS[s] for s in SEEDS], DISTILL["alpha"], DISTILL["T"])
                    if use_kd else base_loss(CFG))
            if not train(student, s_cfg, name, loss):
                continue
            cp.save_full(student.eval(), f)
        cache_variant(name, lambda img, m=student: inf.predict_probs(m, img))
        label = (f"{DISTILL['encoder']} student, distilled from 3-seed ensemble" if use_kd
                 else f"{DISTILL['encoder']} student, annotations only (control)")
        add_row(label, student, name, lambda img, m=student: inf.predict_probs(m, img))
COMPDF = pd.DataFrame(COMP)
display(COMPDF.drop(columns=["note"], errors="ignore").round(4))

## 6b. Fair comparison: post-processing tuned for each source

Sections 5 and 6 scored every probability source and compressed model with the post-processing chain tuned on the single model's outputs. That is not a fair test: averaged probabilities (TTA, ensembles, a distilled student) are smoother and less confident, so the learned instance filter, trained on the single model's instances, removes too many of their filaments. Here each source gets its own cross-fitted chain, with the steps that helped in section 4 (threshold and minimum area, fragment joining, learned filter); the other steps were switched off by the full chain anyway. Scores remain out-of-fold.

In [ ]:
SHORT_STEPS = [STEPS[0], STEPS[1],
               dict(name="+ join fragments", grid=dict(join_r=[0, 4, 8], join_thr=[None, 0.2])),
               STEPS[6]]
SOURCES = {"single model": [f"s{s}" for s in SEEDS]}
if DO_TTA:
    SOURCES["TTA x8"] = [f"s{s}_tta8" for s in SEEDS]
if ENSEMBLE:
    SOURCES["3-seed ensemble"] = ["ensemble"]
if ENSEMBLE and DO_TTA:
    SOURCES["ensemble + TTA x8"] = ["ensemble_tta8"]
for f in sorted(MODEL_DIR.glob("*.pt")):
    SOURCES[f.stem] = [f.stem]

def tune_name(dirs):
    return f"s{TUNE_SEED}" + dirs[0][len(f"s{SEEDS[0]}"):] if len(dirs) > 1 else dirs[0]

FAIR_CHAINS, rows = {}, []
if FAIR_RETUNE:
    for src, dirs in SOURCES.items():
        if not all((PROBS / d).exists() for d in dirs):
            print("not cached, skipped:", src); continue
        FAIR_CHAINS[src] = run_chain(PROBS / tune_name(dirs), "short_" + tune_name(dirs), SHORT_STEPS)
        for i, r in enumerate(FAIR_CHAINS[src]):
            res = [ee.apply_chain(PROBS / d, VAL, CACHE_DIR, FOLDS, r, n_jobs=N_JOBS)[0] for d in dirs]
            rows.append(dict(source=src, step=r["name"], **ee.mean_std(res)))
    FAIR = pd.DataFrame(rows)
    FAIR.to_csv(TAB / "fair_comparison_steps.csv", index=False)
    FAIR_FINAL = FAIR.groupby("source", sort=False).tail(1).drop(columns="step")
    FAIR_FINAL.to_csv(TAB / "fair_comparison.csv", index=False)
    display(FAIR_FINAL[["source", "PQ", "PQ_std", "SQ", "RQ", "TP", "FP", "FN", "one_to_many", "many_to_one"]].round(4))
    display(FAIR.pivot(index="source", columns="step", values="PQ").loc[list(FAIR_CHAINS)][[st["name"] for st in SHORT_STEPS]].round(4))

## 7. Choosing the final pipeline

Every candidate with its out-of-fold PQ (tuning seed) and its end-to-end time per test image, each with the post-processing tuned for it (section 6b). **FINAL** is the most accurate pipeline; **FAST** is the most accurate one whose network is smaller than notebook 02's (a pruned or distilled model). Which one to submit depends on how the competition weighs efficiency (check the Evaluation page); set `CHOICE` below to override.

In [ ]:
def own(src, fallback):
    "The source's own tuned chain (section 6b) if there is one."
    return FAIR_CHAINS[src][-1] if src in FAIR_CHAINS else fallback

CANDIDATES = {
    "single model (nb 02 post-processing)": dict(models=[base_m], tta=1, chain=CHAINS["single model"][0]),
    "single model + enhanced post-processing": dict(models=[base_m], tta=1, chain=CHAINS["single model"][-1]),
}
if DO_TTA:
    CANDIDATES["TTA x8 + enhanced"] = dict(models=[base_m], tta=8, chain=own("TTA x8", CHAINS["single model"][-1]))
if ENSEMBLE:
    CANDIDATES["3-seed ensemble + enhanced"] = dict(models=[MODELS[s] for s in SEEDS], tta=1,
                                                    chain=own("3-seed ensemble", CHAINS.get("3-seed ensemble", CHAINS["single model"])[-1]))
if ENSEMBLE and DO_TTA:
    CANDIDATES["ensemble + TTA x8 + enhanced"] = dict(models=[MODELS[s] for s in SEEDS], tta=8,
                                                      chain=own("ensemble + TTA x8", CHAINS.get("ensemble + TTA x8", CHAINS["single model"])[-1]))
for f in sorted(MODEL_DIR.glob("*.pt")):
    m = cp.load_full(f, DEVICE)
    CANDIDATES[f"{f.stem} + enhanced"] = dict(models=[m], tta=1, chain=own(f.stem, CHAINS["single model"][-1]), probs=f.stem)

prob_name = {"single model (nb 02 post-processing)": f"s{TUNE_SEED}", "single model + enhanced post-processing": f"s{TUNE_SEED}",
             "TTA x8 + enhanced": f"s{TUNE_SEED}_tta8", "3-seed ensemble + enhanced": "ensemble",
             "ensemble + TTA x8 + enhanced": "ensemble_tta8"}
rows = []
for label, c in CANDIDATES.items():
    pdir = PROBS / c.get("probs", prob_name.get(label, ""))
    s, _ = ee.apply_chain(pdir, VAL, CACHE_DIR, FOLDS, c["chain"], n_jobs=N_JOBS)
    blabel = label + (" (own tuning)" if FAIR_CHAINS and "nb 02" not in label else "")
    t = bench(blabel, lambda img, c=c: inf.predict_probs(c["models"], img, tta=c["tta"]), c["chain"]["params"]["all"])
    rows.append(dict(pipeline=label, PQ=s["PQ"], SQ=s["SQ"], RQ=s["RQ"], FP=s["FP"], FN=s["FN"],
                     params_M=sum(cp.count_params(m) for m in c["models"]) / 1e6, **t))
CAND = pd.DataFrame(rows).sort_values("PQ", ascending=False)
CAND.to_csv(TAB / "candidates.csv", index=False)
display(CAND.round(4))
ee.plot_pareto(CAND, FIG / "pq_vs_time.png");

base_time = CAND.set_index("pipeline").total_s["single model (nb 02 post-processing)"]
FINAL = CAND.pipeline.iloc[0]
small = CAND[CAND.params_M < cp.count_params(base_m) / 1e6 - 1e-6]
FAST = small.pipeline.iloc[0] if len(small) else CAND[CAND.total_s <= 1.05 * base_time].pipeline.iloc[0]
CHOICE = None          # e.g. "3-seed ensemble + enhanced", to override the automatic choice
FINAL = CHOICE or FINAL
print("FINAL:", FINAL, "| FAST:", FAST)

## 8. Test predictions and submission

In [ ]:
def predict_test(label):
    c = CANDIDATES[label]
    kw = c["chain"]["params"]["all"]
    out = OUT / "test_pred" / label.replace(" ", "_").replace("+", "").replace("(", "").replace(")", "")
    out.mkdir(parents=True, exist_ok=True)
    for p in tqdm(TEST_PATHS, desc=label):
        f = out / f"{p.stem}.npz"
        if not f.exists():
            img, disk = preprocess_raw(p)
            inst = pp.enhanced(inf.predict_probs(c["models"], img, tta=c["tta"]), disk, **kw)
            np.savez_compressed(f, inst=inst.astype(np.uint16))
    return out

PRED_DIRS = {lab: predict_test(lab) for lab in dict.fromkeys([FINAL, FAST])}
print({k: str(v) for k, v in PRED_DIRS.items()})

In [ ]:
# Submission format: CHECK THIS against the competition's Evaluation page / sample submission.
samples = sorted(Path("/kaggle/input").glob("**/*ample*ubmission*.csv")) if ON_KAGGLE else []
SAMPLE = pd.read_csv(samples[0]) if samples else None
if SAMPLE is not None:
    print(samples[0]); display(SAMPLE.head())
else:
    print("No sample submission found: set FORMAT from the Evaluation page.")

FORMAT = dict(
    id_col="image_id",     # column with the test image name (file stem, e.g. 20110120105534Ch)
    mask_col="rle",        # column with the encoded mask
    instance_col=None,     # e.g. "instance_id" if each filament has its own numbered row
    per="instance",        # "instance": one row per filament; "image": one row per image
    encoding="rle",        # "rle" (start length pairs, 1-indexed) or "coco_rle" (pycocotools string)
    order="C",             # pixel order for "rle": "C" row by row, "F" column by column
    empty="",              # value for an image without filaments (omit the key to write no row)
)
for lab, d in PRED_DIRS.items():
    name = "submission.csv" if lab == FINAL else "submission_fast.csv"
    df = sub.write_submission(d, OUT / name, FORMAT)
    print(name, len(df), "rows")
    if SAMPLE is not None:
        print(sub.check_against_sample(df, SAMPLE) or "matches the sample's columns and ids")
if ON_KAGGLE:
    shutil.copy(OUT / "submission.csv", "/kaggle/working/submission.csv")

## 9. Summary and export

In [ ]:
abl = pd.read_csv(TAB / "ablation_postprocessing.csv")
cand = CAND.set_index("pipeline")
summary = dict(
    model=RUN_NAME, seeds=list(SEEDS), tune_seed=TUNE_SEED, folds=int(max(FOLDS.values()) + 1),
    baseline_PQ=float(abl.PQ.iloc[0]), baseline_PQ_std=float(abl.PQ_std.iloc[0]),
    postprocessing_PQ=float(abl.PQ.iloc[-1]), postprocessing_PQ_std=float(abl.PQ_std.iloc[-1]),
    final_pipeline=FINAL, final_PQ=float(cand.PQ[FINAL]), final_total_s=float(cand.total_s[FINAL]),
    fast_pipeline=FAST, fast_PQ=float(cand.PQ[FAST]), fast_total_s=float(cand.total_s[FAST]),
    final_params=ee._show(CANDIDATES[FINAL]["chain"]["params"]["all"]),
    device=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
    hours=(time.time() - T_START) / 3600,
)
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
print(json.dumps(summary, indent=1, default=str))

# One zip with everything except model weights and training checkpoints: tables, figures, chosen
# parameters, test-set instance maps and submission files (for the slides and results/enhancement).
import zipfile
with zipfile.ZipFile(OUT.parent / "enhancement_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file() and f.suffix not in (".pt", ".pkl"):
            z.write(f, f.relative_to(OUT.parent))
print("download:", OUT.parent / "enhancement_results.zip")